<a href="https://colab.research.google.com/github/akashgoyalll/finsight-rag-I025-39-40/blob/main/FinSight_RAG_Phase3_executed_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight-RAG — Phase 3 (Final System)
**A Source-Grounded Question-Answering Assistant for Corporate Financial Filings**
Lab 9 PSIS · Activity 1 (RAG-Based Domain Assistant) · Akash Goyal (I025) · Karan Lokhande (I039) · Meer Magia (I040)

**Before you start:** add your Gemini key in Colab: click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* →
name `GOOGLE_API_KEY`, value = your key from Google AI Studio → turn **Notebook access** on. Then **Runtime → Run all**.
The key is read from Colab Secrets and is never printed or saved to the repository.

What this notebook does:
1. Clones the repo, installs the pinned packages and downloads the NIKE FY2023 10-K (107 pages).
2. Runs the full Phase 3 evaluation: semantic vs keyword vs **hybrid** retrieval, answers from Gemini for Q1–Q7
   (Phase 2 baseline vs Phase 3 final system), automatic checks, and the multi-turn follow-up test.
3. Prints every answer next to the full text of the passage it cites, for the manual grounding review.
4. Live demo cells (used in the video) and the test suite.

## 1. Setup

In [ ]:
%cd /content
!test -d finsight-rag-I025-39-40 || git clone -q https://github.com/akashgoyalll/finsight-rag-I025-39-40.git
%cd /content/finsight-rag-I025-39-40
!git pull -q
# If the Phase 3 code is not on GitHub yet: drag finsight-rag-phase3.zip into the Files panel first
# (any folder under /content; a renamed download such as "finsight-rag-phase3 (1).zip" is fine).
import glob, os, subprocess
zips = sorted(glob.glob("/content/**/finsight-rag-phase3*.zip", recursive=True), key=os.path.getmtime)
if zips:
    subprocess.run(["unzip", "-o", "-q", zips[-1], "-d", "."], check=True)
    print("Applied Phase 3 files from", zips[-1])
!git log --oneline -3
if not os.path.exists("scripts/run_phase3.py"):
    raise SystemExit("STOP: Phase 3 code missing. Push it to GitHub, or upload finsight-rag-phase3.zip in the Files panel, then re-run this cell.")
print("Phase 3 code present")
!pip install -q -r requirements.txt
!bash scripts/get_data.sh
!ls -la data

/content
/content/finsight-rag-I025-39-40
1fbb546 (HEAD -> main, origin/main, origin/HEAD) Merge branch 'main' of https://github.com/akashgoyalll/finsight-rag-I025-39-40
d76dd56 update
2141da9 requirements
Phase 3 code present
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2341k  100 2341k    0     0  22.6M      0 --:--:-- --:--:-- --:--:-- 22.8M
total 2352
drwxr-xr-x  2 root root    4096 Oct  1 17:37 .
drwxr-xr-x 10 root root    4096 Oct  1 17:37 ..
-rw-r--r--  1 root root 2397936 Oct  1 17:39 nke-10k-2023.pdf


## 2. Gemini API key and model

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API key: ")
# Free-tier Gemini model (Google AI Studio). If you get repeated 429 (rate-limit) errors, switch to
# "google_genai:gemini-3.5-flash-lite" and re-run from here.
os.environ["LLM_MODEL"] = "google_genai:gemini-3.5-flash"
print("Key loaded:", bool(os.environ.get("GOOGLE_API_KEY")), "| model:", os.environ["LLM_MODEL"])

Gemini API key: ··········
Key loaded: True | model: google_genai:gemini-3.5-flash


## 3. Full Phase 3 evaluation
Retrieval comparison for Q1–Q6 (no LLM), then Gemini answers for Q1–Q7 with the **Phase 2 baseline** (semantic, top-4)
and the **Phase 3 system** (hybrid semantic + keyword with reciprocal rank fusion, top-6), then the two-turn memory test
with and without follow-up rewriting. Takes about 5–10 minutes (index build + ~20 LLM calls with a 4-second pause between calls).
Results are saved to `outputs/phase3_results.json`, `outputs/phase3_summary.md` and `outputs/phase3_log_colab.txt`.

In [ ]:
import os, subprocess, sys
# Runs the evaluation, streams its output here and saves it to outputs/phase3_log_colab.txt
with open("outputs/phase3_log_colab.txt", "w") as log:
    p = subprocess.Popen([sys.executable, "scripts/run_phase3.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "unauthenticated requests" in line or "it/s]" in line:
            continue
        print(line, end=""); log.write(line)
    p.wait()
if p.returncode != 0 or not os.path.exists("outputs/phase3_summary.md"):
    raise SystemExit(f"STOP: run_phase3.py failed (exit code {p.returncode}). The error is in the last lines printed above.")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Pages 107 | chunks 516 | semantic index = hf | keyword index = tfidf | built in 89.1s

=== Retrieval comparison (Q1-Q6). string = evidence string in retrieved text; page = a ground-truth page retrieved ===
semantic@4  string 4/6  page 5/6  | Q1:H  Q2:m  Q3:H  Q4:H  Q5:m  Q6:H
keyword@4   string 4/6  page 5/6  | Q1:m  Q2:H  Q3:H  Q4:H  Q5:H  Q6:m
semantic@6  string 4/6  page 6/6  | Q1:H  Q2:m  Q3:H  Q4:H  Q5:m  Q6:H
keyword@6   string 4/6  page 5/6  | Q1:m  Q2:H  Q3:H  Q4:H  Q5:H  Q6:m
hybrid@4    string 6/6  page 6/6  | Q1:H  Q2:H  Q3:H  Q4:H  Q5:H  Q6:H
hybrid@6    string 6/6  page 6/6  | Q1:H  Q2:H  Q3:H  Q4:H  Q5:H  Q6:H

LLM: google_genai:gemini-3.5-flash

=== Answers: final (Phase 3: hybrid@

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/phase3_summary.md").read()))

# Phase 3 results

Model: `google_genai:gemini-3.5-flash` · semantic backend: `hf` · 107 pages, 516 chunks · generated 2026-10-01 17:45:03 +0000

## Retrieval (Q1–Q6)

| Retriever | Q1 | Q2 | Q3 | Q4 | Q5 | Q6 | string hits | page hits |
|---|---|---|---|---|---|---|---|---|
| semantic@4 | HIT [36, 89, 31, 32] | miss [4, 86, 6, 36] | HIT [37, 38, 47, 31] | HIT [36, 36, 31, 43] | miss [46, 46, 82, 88] | HIT [39, 39, 39, 32] | 4/6 | 5/6 |
| keyword@4 | miss [42, 43, 41, 40] | HIT [28, 9, 79, 9] | HIT [37, 38, 48, 47] | HIT [36, 31, 41, 40] | HIT [14, 41, 14, 32] | miss [32, 39, 37, 39] | 4/6 | 5/6 |
| semantic@6 | HIT [36, 89, 31, 32, 4, 35] | miss [4, 86, 6, 36, 89, 9] | HIT [37, 38, 47, 31, 38, 47] | HIT [36, 36, 31, 43, 37, 89] | miss [46, 46, 82, 88, 45, 46] | HIT [39, 39, 39, 32, 31, 36] | 4/6 | 6/6 |
| keyword@6 | miss [42, 43, 41, 40, 6, 44] | HIT [28, 9, 79, 9, 78, 96] | HIT [37, 38, 48, 47, 47, 38] | HIT [36, 31, 41, 40, 5, 6] | HIT [14, 41, 14, 32, 13, 45] | miss [32, 39, 37, 39, 87, 31] | 4/6 | 5/6 |
| hybrid@4 | HIT [36, 42, 89, 43] | HIT [4, 28, 86, 9] | HIT [37, 38, 47, 48] | HIT [36, 31, 36, 41] | HIT [46, 14, 46, 41] | HIT [32, 39, 39, 39] | 6/6 | 6/6 |
| hybrid@6 | HIT [36, 42, 89, 43, 31, 41] | HIT [4, 28, 86, 9, 6, 79] | HIT [37, 38, 47, 48, 31, 47] | HIT [36, 31, 36, 41, 43, 40] | HIT [46, 14, 46, 41, 82, 14] | HIT [32, 39, 39, 39, 37] | 6/6 | 6/6 |

## Answers — final (Phase 3: hybrid@6)

| Q | answer | answerable | cited pages | answer_ok | cited_ok |
|---|---|---|---|---|---|
| Q1 | In fiscal 2023, NIKE, Inc.'s total revenues were $51.2 billion (or $51,217 million). | True | [36, 89, 31] | True | True |
| Q2 | As of May 31, 2023, NIKE had approximately 83,700 employees worldwide, including retail and part-time employees. | True | [9] | True | True |
| Q3 | For fiscal 2023, gross margin decreased 250 basis points to 43.5% compared to 46.0% for fiscal 2022. | True | [37] | True | True |
| Q4 | In fiscal 2023, NIKE Direct revenues grew 14% (or 20% on a currency-neutral basis) from $18.7 billion in fiscal 2022 to $21.3 billion in fiscal 2023. | True | [36, 31] | True | True |
| Q5 | Based on the provided filings, NIKE faces several risks from fluctuations in foreign currency exchange rates:  1. **Financial Impact on Revenues, Costs, and Earnings:** Fluctuations can result in lower revenues, higher costs, and decreased margins and earnings. Specifically, a weakening of foreign currencies relative to the U.S. Dollar adversely affects the U.S. Dollar value of NIKE's foreign currency-denominated sales and earnings [2, 6]. Overall, these fluctuations can have an adverse effect on results of operations and financial condition [6]. 2. **Impact on Product and Inventory Costs:** NIKE is subject to transactional foreign currency risk where product costs are exposed to fluctuations [3]. Under its factory currency exposure program, a weakening U.S. Dollar against indexed foreign currencies increases NIKE's U.S. Dollar inventory costs (though a strengthening U.S. Dollar reduces it) [1]. 3. **Disruption to Independent Manufacturers:** Currency exchange rate fluctuations can disrupt the business of independent manufacturers that produce NIKE's products by making their purchases of raw materials more expensive and more difficult to finance [6]. | True | [46, 14, 46, 14] | None | True |
| Q6 | Based on the data provided for fiscal 2023, Greater China had the highest EBIT margin among the NIKE Brand geographic segments at approximately 31.50%.  Using the EBIT margin definition (EBIT divided by Revenues), the segment EBIT margins for fiscal 2023 are calculated as follows:  * **Greater China**: EBIT of $2,283 million divided by Revenues of $7,248 million ≈ 31.50% * **Asia Pacific & Latin America**: EBIT of $1,932 million divided by Revenues of $6,431 million ≈ 30.04% * **Europe, Middle East & Africa**: EBIT of $3,531 million divided by Revenues of $13,418 million ≈ 26.31% * **North America**: EBIT of $5,454 million divided by Revenues of $21,608 million ≈ 25.24% | True | [32, 39, 39, 39] | True | True |
| Q7 | I am sorry, but the provided excerpts do not contain information about NIKE's total revenues in fiscal 2025. The data only goes up to fiscal 2023. | False | [] | True | None |

## Multi-turn follow-up (expected: 16% currency-neutral)

- **without question rewriting (Phase 2)** — retrieval query: “How much did that grow on a currency-neutral basis?” → In fiscal 2023, NIKE, Inc. revenues increased 16% compared to fiscal 2022 on a currency-neutral basis. (answer_ok=True, history messages=4)
- **with question rewriting (Phase 3)** — retrieval query: “None” → GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}} (answer_ok=False, history messages=2)


## 4. Manual grounding review
For each answer of the final system: the answer, then the **full text of every chunk it cites** (document, page).
Grounded = every figure/claim in the answer appears in (or is directly computed from) the cited text.

In [ ]:
import os, sys
REPO = "/content/finsight-rag-I025-39-40"
os.chdir(REPO); sys.path.insert(0, f"{REPO}/src")
for m in [m for m in sys.modules if m == "finsight" or m.startswith("finsight.")]: del sys.modules[m]
import finsight.indexing; print("OK:", finsight.indexing.__file__)

OK: /content/finsight-rag-I025-39-40/src/finsight/indexing.py


In [ ]:
import json, sys, textwrap
sys.path.insert(0, "src")
from finsight.indexing import load_pages, chunk
from finsight.hybrid import chunk_key
chunks = chunk(load_pages(["data/nke-10k-2023.pdf"]))
by_key = {chunk_key(c): c for c in chunks}
res = json.load(open("outputs/phase3_results.json"))
final = [k for k in res["answers"] if k.startswith("final")][0]
for r in res["answers"][final]:
    print("=" * 100)
    print(f"{r['id']}  {r['question']}\nTRUTH : {r['ground_truth']}\nANSWER: {r['answer'] or r.get('error')}")
    print(f"answerable={r['answerable']}  auto answer_ok={r['answer_ok']}  cited_ok={r['cited_ok']}")
    for s in r["sources"]:
        c = by_key.get((s["source"], s["page"], s["start_index"], s["passage"][:50]))
        print(f"  --- cited [{s['excerpt']}] {s['source']} p.{s['page']} ---")
        print(textwrap.indent(textwrap.fill((c.page_content if c else s["passage"]), 110), "    "))

Q1  What were NIKE, Inc.'s total revenues in fiscal 2023?
TRUTH : $51.2 billion ($51,217 million), +10% reported / +16% currency-neutral
ANSWER: In fiscal 2023, NIKE, Inc.'s total revenues were $51.2 billion (or $51,217 million).
answerable=True  auto answer_ok=True  cited_ok=True
  --- cited [1] nke-10k-2023.pdf p.36 ---
    Table of Contents FISCAL 2023 NIKE BRAND REVENUE HIGHLIGHTSThe following tables present NIKE Brand revenues
    disaggregated by reportable operating segment, distribution channel and major product line: FISCAL 2023
    COMPARED TO FISCAL 2022 • NIKE, Inc. Revenues were $51.2 billion in fiscal 2023, which increased 10% and 16%
    compared to fiscal 2022 on a reported and currency-neutral basis, respectively. The increase was due to higher
    revenues in North America, Europe, Middle East & Africa ("EMEA"), APLA and Greater China, which contributed
    approximately 7, 6, 2 and 1 percentage points to NIKE, Inc. Revenues, respectively. • NIKE Brand revenues,
    w

## 5. Live demo (used in the video)
Builds the final system in memory (hybrid retrieval + Gemini + memory) and answers questions with page citations.

In [ ]:
import warnings
from langchain_core._api import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
from finsight.indexing import index_chunks
from finsight.hybrid import build_hybrid_retriever
from finsight.chain import build_chain, with_memory
from finsight.llm import get_llm

retriever = build_hybrid_retriever(index_chunks(chunks, kind="hf"), index_chunks(chunks, kind="tfidf"), fetch_k=4, k=6)
chat = with_memory(build_chain(retriever, get_llm()))

def ask(question, session="demo"):
    out = chat.invoke({"question": question}, config={"configurable": {"session_id": session}})
    print(f"Q: {question}")
    if out["retrieval_query"] != question:
        print(f"   (rewritten for retrieval: {out['retrieval_query']})")
    print(f"A: {out['answer']}\n   answerable = {out['answerable']}")
    for s in out["sources"]:
        print(f"   source [{s['excerpt']}]: {s['source']}, page {s['page']}  |  {s['passage'][:120].replace(chr(10), ' ')}...")
    print(f"   retrieved pages: {out['retrieved_pages']}\n")
    return out

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
_ = ask("How many employees did NIKE have as of May 31, 2023?", session="demo-1")

Q: How many employees did NIKE have as of May 31, 2023?
A: As of May 31, 2023, NIKE had approximately 83,700 employees worldwide, including retail and part-time employees.
   answerable = True
   source [4]: nke-10k-2023.pdf, page 9  |  organizations through donations and volunteering, which are matched by the NIKE Foundation where eligible. EMPLOYEE BASE...
   retrieved pages: [4, 28, 86, 9, 6, 79]



In [ ]:
_ = ask("What were NIKE, Inc.'s total revenues in fiscal 2023?", session="demo-2")
_ = ask("How much did that grow on a currency-neutral basis?", session="demo-2")   # follow-up uses memory

Q: What were NIKE, Inc.'s total revenues in fiscal 2023?
A: In fiscal 2023, NIKE, Inc.'s total revenues were $51.2 billion (or $51,217 million).
   answerable = True
   source [1]: nke-10k-2023.pdf, page 36  |  Table of Contents FISCAL 2023 NIKE BRAND REVENUE HIGHLIGHTSThe following tables present NIKE Brand revenues disaggregate...
   source [3]: nke-10k-2023.pdf, page 89  |  Table of Contents YEAR ENDED MAY 31, (Dollars in millions) 2023 2022 2021 REVENUES North America $ 21,608 $ 18,353 $ 17,...
   source [5]: nke-10k-2023.pdf, page 31  |  Enterprise Resource Planning Platform, data and analytics, demand sensing, insight gathering, and other areas to create ...
   retrieved pages: [36, 42, 89, 43, 31, 41]

Q: How much did that grow on a currency-neutral basis?
   (rewritten for retrieval: How much did NIKE, Inc.'s total revenues grow on a currency-neutral basis in fiscal 2023?)
A: On a currency-neutral basis, NIKE, Inc.'s revenues increased 16% in fiscal 2023 compared to fiscal 2022

In [ ]:
_ = ask("Which NIKE Brand geographic segment had the highest EBIT margin in fiscal 2023?", session="demo-3")  # weak case

Q: Which NIKE Brand geographic segment had the highest EBIT margin in fiscal 2023?
A: To find the NIKE Brand geographic segment with the highest EBIT margin in fiscal 2023, we divide each segment's EBIT by its revenues:

1. **Greater China**:
   * EBIT: $2,283 million
   * Revenues: $7,248 million
   * EBIT Margin: $2,283 / $7,248 ≈ 31.50%

2. **Asia Pacific & Latin America**:
   * EBIT: $1,932 million
   * Revenues: $6,431 million
   * EBIT Margin: $1,932 / $6,431 ≈ 30.04%

3. **Europe, Middle East & Africa**:
   * EBIT: $3,531 million
   * Revenues: $13,418 million
   * EBIT Margin: $3,531 / $13,418 ≈ 26.32%

4. **North America**:
   * EBIT: $5,454 million
   * Revenues: $21,608 million
   * EBIT Margin: $5,454 / $21,608 ≈ 25.24%

Therefore, **Greater China** had the highest EBIT margin in fiscal 2023.
   answerable = True
   source [2]: nke-10k-2023.pdf, page 39  |  North America $ 5,454 $ 5,114 7 % $ 5,089 0 % Europe, Middle East & Africa 3,531 3,293 7 % 2,435 35 % Greater China 2,

In [ ]:
_ = ask("What were NIKE's total revenues in fiscal 2025?", session="demo-4")  # not in the filing -> should refuse

Q: What were NIKE's total revenues in fiscal 2025?
A: I am sorry, but the provided excerpts do not contain information about NIKE's total revenues in fiscal 2025.
   answerable = False
   retrieved pages: [36, 42, 89, 43, 32, 41]



## 6. Test suite (offline, no API calls)

In [ ]:
!python -m pytest tests/ -v -p no:cacheprovider -W ignore::DeprecationWarning 2>&1 | tee outputs/phase3_pytest_log_colab.txt | tail -16

plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collecting ... collected 11 items

tests/test_pipeline.py::test_loader_attaches_source_and_page PASSED      [  9%]
tests/test_pipeline.py::test_metadata_survives_chunking PASSED           [ 18%]
tests/test_pipeline.py::test_retriever_returns_cited_chunks PASSED       [ 27%]
tests/test_pipeline.py::test_parser_rejects_malformed_output PASSED      [ 36%]
tests/test_pipeline.py::test_sources_resolved_from_metadata_not_llm PASSED [ 45%]
tests/test_pipeline.py::test_full_lcel_chain_and_memory_wiring PASSED    [ 54%]
tests/test_pipeline.py::test_rrf_fuses_ranked_lists_and_dedupes PASSED   [ 63%]
tests/test_pipeline.py::test_hybrid_retriever_is_a_langchain_retriever PASSED [ 72%]
tests/test_pipeline.py::test_rebuilding_index_does_not_duplicate_chunks PASSED [ 81%]
tests/test_pipeline.py::test_followup_question_is_rewritten_before_retrieval PASSED [ 90%]
tests/test_pipeline.py::test_llm_factory_sampling_params PASSED          [100%]

===

## 7. Download the results
Downloads `phase3_outputs.zip` (results JSON, summary, logs). Send it to the report owner / commit it to `outputs/`.

In [ ]:
!cd outputs && zip -q -j ../phase3_outputs.zip phase3_results.json phase3_summary.md phase3_log_colab.txt phase3_pytest_log_colab.txt
from google.colab import files
files.download("phase3_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>